# T05 · Modélisation statistique — ADIF72

**Objectifs :**
- Distributions d'échantillonnage
- Théorème de la Limite Centrale (TCL)
- Estimation : biais, variance, MSE
- Intervalles de confiance (moyenne, proportion)
- Régression linéaire & corrélation

**Lien avec T04 :** si `X` a une loi (mêlée), la moyenne d'un échantillon
a **aussi** une loi (distribution d'échantillonnage). C'est ce qui permet
d'inférer (T06) et de construire des intervalles de confiance.

In [ ]:
import numpy as np
import pandas as pd
from scipy import stats
import sklearn.linear_model as lm
np.set_printoptions(precision=4, suppress=True, linewidth=100)
print("Prêt. sklearn", lm.LinearRegression.__module__)

## 1 · Distribution d'échantillonnage

Si `X ~ (μ, σ²)`, et `X₁...Xₙ` i.i.d., alors la **moyenne d'échantillon** :
```
X̄ = (1/n) Σ Xᵢ   ~   (μ, σ²/n)
```
C'est-à-dire : `E[X̄] = μ` et `Var(X̄) = σ²/n`.
Conséquence clé : **l'erreur standard diminue en `1/√n`**. Pour
diviser par 2 l'erreur, il faut 4× plus de données. C'est le coût
de la précision statistique.

On peut aussi écrire `X̄ ~ N(μ, σ²/n)` **si** la population est normale ;
sinon, le TCL (suivant) garantit la normalité **asymptotique**.

In [ ]:
mu, sigma, n = 10.0, 3.0, 25
rng = np.random.default_rng(0)
means = [rng.normal(mu, sigma, n).mean() for _ in range(2000)]
means = np.array(means)
print(f"Population N({mu},{sigma}) ; échantillons n={n}")
print(f"Moyenne des X̄ = {means.mean():.4f}  (≈ {mu})")
print(f"SD des X̄      = {means.std(ddof=1):.4f}  (≈ {sigma/np.sqrt(n):.4f})")
assert abs(means.mean() - mu) < 0.1
assert abs(means.std(ddof=1) - sigma/np.sqrt(n)) < 0.1

### Exercice corrigé

In [ ]:
# ---- Exercice P1 : loi de Xbar pour X~N(5,2), n=100 ----
print(f"P1-ex : E[Xbar] = 5, Var[Xbar] = 4/100 = {4/100}")
assert np.isclose(4/100, 0.04)


## 2 · Théorème de la Limite Centrale (TCL)

**Énoncé (simplifié)** : si `X₁,...,Xₙ` sont i.i.d. avec moyenne `μ`
et variance finie `σ²`, alors :
```
Z_n = (X̄ - μ) / (σ/√n)  --(n→∞)-->  N(0,1)
```
**C'est le théorème le plus important de la statistique.** Il dit que :
- La moyenne d'échantillons devient **normale**, peu importe la loi source
- La vitesse de convergence est en `O(1/√n)`

Applications directes :
- **Intervalles de confiance** (section 4)
- **Tests z** (T06)
- Justification du Gaussian noise dans les réseaux de neurones

In [ ]:
# Population uniforme [0,1] (pas normale !) -> la moyenne est quasi normale
n = 30
mu_u, var_u = 0.5, 1/12
rng = np.random.default_rng(0)
Z = [(rng.uniform(0, 1, n).mean() - mu_u) / np.sqrt(var_u / n)
     for _ in range(5000)]
Z = np.array(Z)
print(f"Tirages de la moyenne de {n} uniformes, standardisés")
print(f"  E[Z]  = {Z.mean():.4f}  (attendu 0)")
print(f"  Var[Z]= {Z.var():.4f}  (attendu 1)")
print(f"  kurtosis (excès) = {stats.kurtosis(Z):.4f}  (normale: 0)")
w, p = stats.shapiro(Z[:500])
print(f"  Shapiro-Wilk p = {p:.3f}  (>0.05 => compatible normalité)")
assert abs(Z.mean()) < 0.05 and abs(Z.var() - 1) < 0.1

### Exercice corrigé

In [ ]:
# ---- Exercice P2 : P(Xbar > 105), population N(100,15), n=9 ----
p = 1 - stats.norm.cdf(105, 100, 15/3)
print(f"P2-ex : Xbar ~ N(100, 5^2), P(Xbar > 105) = {p:.4f}  (= 1-Phi(1))")
assert np.isclose(p, 0.1587, atol=1e-3)


## 3 · Estimation — biais & variance d'un estimateur

Un **estimateur** est une règle (fonction des données) qui produit une
valeur pour un paramètre inconnu (ici `μ`).

Mesures de qualité :
```
Biais(θ̂) = E[θ̂] - θ
Var(θ̂)   = Var(θ̂)
MSE(θ̂)   = Biais² + Var(θ̂)
```
- **Sans biais** : `E[θ̂] = θ` (pas de dérive moyenne)
- **Efficient** : parmi les estimateurs sans biais, le plus petite variance

L'exemple `moyenne vs 1re observation` montre que deux estimateurs
peuvent être sans biais, mais l'un est **beaucoup plus précis**
(variance `σ²/n` vs `σ²`).

In [ ]:
mu, sigma = 10.0, 3.0
def bias_variance(estimator, n_trials=20000, rng=np.random.default_rng(1)):
    vals = np.array([estimator(rng) for _ in range(n_trials)])
    return vals.mean() - mu, vals.var(), vals

est_mean = lambda r: r.normal(mu, sigma, size=25).mean()
est_first = lambda r: r.normal(mu, sigma, size=1)[0]
bm, vm, _ = bias_variance(est_mean)
bf, vf, _ = bias_variance(est_first)
print(f"moyenne (n=25) : bias={bm:+.4f}, var={vm:.4f}, MSE={bm**2+vm:.4f}")
print(f"1re obs        : bias={bf:+.4f}, var={vf:.4f}, MSE={bf**2+vf:.4f}")
print(f"Var de moyenne / σ² = {vm/sigma**2:.4f} (= 1/n)")
assert abs(bm) < 0.02 and vm < vf

### Exercice corrigé

In [ ]:
# ---- Exercice P3 : Var(Xbar, n=25) vs Var((X1+X2)/2) ----
print(f"P3-ex : Var(Xbar n=25) = 1/25 = {1/25}")
print(f"       Var((X1+X2)/2) = 1/4*(1+1) = {0.25*(1+1)}")
assert np.isclose(1/25, 0.04) and np.isclose(0.25*(1+1), 0.5)


## 4 · Intervalles de confiance (IC)

Un **IC à 95%** pour un paramètre est un intervalle `[a, b]` construit
à partir des données tel que :
- On dit « le paramètre **est** dans `[a,b]` avec 95% de confiance ».
- **Interprétation correcte** : si on répétait l'expérience 100 fois,
  **95** des intervalles produiraient contiendraient le vrai paramètre.
- Le vrai paramètre est **fixé** (pas aléatoire) — c'est l'intervalle
  qui varie. C'est le piège n°1 à l'examen.

**Cas 1** — variance connue : `IC = X̄ ± z₀.₉₇₅ · σ/√n`
**Cas 2** — variance inconnue : `IC = X̄ ± t₀.₉₇₅,df · s/√n`
(où `s` = écart-type échantillonnal, `df = n-1`)

In [ ]:
mu, sigma = 10.0, 3.0
rng = np.random.default_rng(0)
data = rng.normal(mu, sigma, size=60)
xbar = data.mean(); se = sigma / np.sqrt(len(data))
z975 = stats.norm.ppf(0.975)
ic_mu = (xbar - z975*se, xbar + z975*se)
print(f"n={len(data)}, X̄={xbar:.3f}, σ connue={sigma}")
print(f"IC 95% (normale)  : [{ic_mu[0]:.3f}, {ic_mu[1]:.3f}]  contient μ? {ic_mu[0]<=mu<=ic_mu[1]}")

data2 = rng.normal(mu, sigma, size=60)
xbar2 = data2.mean(); s2 = data2.std(ddof=1); se2 = s2/np.sqrt(len(data2))
t975 = stats.t.ppf(0.975, df=len(data2)-1)
ic_t = (xbar2 - t975*se2, xbar2 + t975*se2)
print(f"IC 95% (t, s inconnue={s2:.3f}): [{ic_t[0]:.3f}, {ic_t[1]:.3f}]  contient μ? {ic_t[0]<=mu<=ic_t[1]}")
print(f"  z={z975:.4f} vs t={t975:.4f} (t plus large)")

### Exercice corrigé

In [ ]:
# ---- Exercice P4 : IC95 pour une proportion (n=100, phat=0.60) ----
se = np.sqrt(0.6*0.4/100)
lo, hi = 0.6 - 1.96*se, 0.6 + 1.96*se
print(f"P4-ex : se = {se:.4f}, IC95 = [{lo:.4f}, {hi:.4f}]")
assert np.isclose(lo, 0.504, atol=1e-3) and np.isclose(hi, 0.696, atol=1e-3)


## 5 · Régression linéaire

### Simple : `y = β₀ + β₁ x + ε`
Les paramètres sont estimés par **moindres carrés** :
```
minimize  Σ (yᵢ - β₀ - β₁ xᵢ)²
```
Solution close-form (rappel T01 — système normal) :
```
β₁ = S_xy / S_xx  =  Σ(xᵢ-x̄)(yᵢ-ȳ) / Σ(xᵢ-x̄)²
β₀ = ȳ - β₁ x̄
R² = 1 - SSE/SST   (fraction de variance expliquée)
```

### Multiple : `y = β₀ + β₁x₁ + ... + βₖxₖ + ε`
Même principe en matriciel : `β̂ = (XᵀX)⁻¹ Xᵀy`.
Si `XᵀX` est mal conditionné → **régularisation** (Ridge, Lasso, ElasticNet)
— on y reviendra au T07.

In [ ]:
rng = np.random.default_rng(42)
x = rng.uniform(0, 10, 100)
y = 1.5 * x + 2 + rng.normal(0, 1, 100)
reg = lm.LinearRegression().fit(x.reshape(-1, 1), y)
pred = reg.predict(x.reshape(-1, 1))
r2 = reg.score(x.reshape(-1, 1), y)
print(f"β₁ estimé = {reg.coef_[0]:.3f} (vrai 1.5),  β₀ = {reg.intercept_:.3f} (vrai 2)")
print(f"R² = {r2:.4f}")
assert abs(reg.coef_[0] - 1.5) < 0.15 and r2 > 0.7

# Régression multiple
Xm = rng.uniform(-3, 3, (150, 2))
beta_true = np.array([2.0, -1.0])
ym = Xm @ beta_true + 0.5 + rng.normal(0, 0.5, 150)
regM = lm.LinearRegression().fit(Xm, ym)
print(f"\nMultiple : coefs = {regM.coef_} (vrais [2,-1]), "
      f"intercepte={regM.intercept_:.3f} (vrai 0.5)")
assert np.allclose(regM.coef_, beta_true, atol=0.15)

### Exercice corrigé

In [ ]:
# ---- Exercice P5 : regression a la main (5 points alignes) ----
xr = np.array([0.0, 1, 2, 3, 4]); yr = np.array([1.0, 3, 5, 7, 9])
beta_ = ((xr-xr.mean())*(yr-yr.mean())).sum() / ((xr-xr.mean())**2).sum()
alpha_ = yr.mean() - beta_ * xr.mean()
print(f"P5-ex : beta = {beta_:.1f}, alpha = {alpha_:.1f}  (R^2 = 1, droite exacte)")
assert beta_ == 2 and alpha_ == 1


## 6 · Corrélation & coefficient de détermination

**Corrélation de Pearson** `r` :
```
r = cov(x,y) / (sd_x · sd_y) ∈ [-1, 1]
```
Mesure de la **linéarité** de la relation entre deux variables.
`r = 0` : pas de relation **linéaire** (il peut y en avoir une non-linéaire !).
`r²` (coefficient de détermination) : fraction de la variance de `y`
expliquée par `x` (dans un modèle linéaire).

**Pièges classiques :**
- `r` ≠ causalité
- `r` sensible aux outliers
- `r = 0` ne signifie pas « indépendance » (seulement « non linéaire »)

In [ ]:
df = pd.DataFrame({'A': rng.normal(0, 1, 50)})
df['B'] = df['A'] * 0.8 + rng.normal(0, 1, 50)
df['C'] = rng.normal(0, 1, 50)
corr_AB = np.corrcoef(df['A'], df['B'])[0, 1]
corr_AC = np.corrcoef(df['A'], df['C'])[0, 1]
print(f"corr(A,B) = {corr_AB:.3f}  (r² = {corr_AB**2:.3f})")
print(f"corr(A,C) = {corr_AC:.3f}  (≈ 0 : indépendantes)")
assert abs(corr_AB) > 0.5 and abs(corr_AC) < 0.3

# Corrélation ≠ indépendance : U-shape
x_u = rng.normal(0, 1, 100)
y_u = x_u**2 + rng.normal(0, 0.5, 100)
print(f"\ncorr(x, x²+bruit) = {np.corrcoef(x_u, y_u)[0,1]:.3f}  (≈ 0, pourtant y dépend de x)")

### Exercice corrigé

In [ ]:
# ---- Exercice P6 : coefficient de correlation ----
x = np.array([1.0, 2, 3, 4]); y = np.array([2.0, 1, 4, 3])
r = np.corrcoef(x, y)[0, 1]
print(f"P6-ex : r(x,y) = {r:.4f}")
assert np.isclose(r, 0.6)


## 7 · Régression linéaire bayésienne *(MML ch. 9.3)*

Le T05 §5 a donné la régression **frequentiste** (maximum de vraisemblance).
La version **bayésienne** traite les poids `w` comme aléatoires :
- **prior** : `w ~ N(0, α⁻¹ I)` (contraints autour de 0 → effet de Ridge)
- **bruit** : `y = Xw + ε`, `ε ~ N(0, β⁻¹ I)`
- **posterior** : `w | D ~ N(m_N, S_N)` avec
```
S_N⁻¹ = α I + β XᵀX
m_N   = S_N (α m₀ + β Xᵀy)
```
- **prédiction** : `y* | x*, D ~ N(m_Nᵀx*, 1/β + x*ᵀ S_N x*)`

**Pourquoi c'est utile :**
1. le prior `α I` = **régularisation Ridge** (liage direct avec T07)
2. le **doubt quantifié** : la variance prédictive croît avec l'écart
   au nuage de données
3. cadre unifié : le MLE correspond au cas de prior uniforme (α→0)

In [ ]:
rng = np.random.default_rng(3)
n = 100
X = rng.normal(0, 1, (n, 1))
beta_true = np.array([1.5])
y = X @ beta_true + 0.0 + rng.normal(0, 0.5, n)

ols = lm.LinearRegression().fit(X, y)
print(f"OLS :  β̂ = {ols.coef_[0]:.3f} (vrai 1.5)")

def bayes_linear_reg(X, y, alpha, beta, m0=None):
    m0 = np.zeros(X.shape[1]) if m0 is None else m0
    S0_inv = alpha * np.eye(X.shape[1])
    S_N_inv = S0_inv + beta * (X.T @ X)
    S_N = np.linalg.inv(S_N_inv)
    m_N = S_N @ (S0_inv @ m0 + beta * (X.T @ y))
    return m_N, S_N

beta_noise = 4.0   # 1/σ² avec σ=0.5
for a, lbl in [(0.1, "prior faible (α=0.1)"), (1000.0, "prior fort (α=1000, sd=0.03)")]:
    m_N, S_N = bayes_linear_reg(X, y, a, beta_noise)
    print(f"{lbl} : β̂_bayes = {m_N[0]:.3f} ± {np.sqrt(S_N[0,0]):.3f}")

# Vérif : prior faible ~ OLS ; prior fort tire vers 0
m_weak, _ = bayes_linear_reg(X, y, 0.1, beta_noise)
m_strong, S_strong = bayes_linear_reg(X, y, 1000.0, beta_noise)
assert abs(m_weak[0] - ols.coef_[0]) < 0.05
assert abs(m_strong[0]) < 0.6   # tiré fortement vers 0 (Ridge extrême)

# Incertitude prédictive à x* = 0 (dans le nuage) vs x* = 3 (loin)
x_near = np.array([[0.0]])
x_far = np.array([[3.0]])
for xx, lbl in [(x_near, "x*=0 (dans le nuage)"), (x_far, "x*=3 (loin)")]:
    pred_var = 1 / beta_noise + (xx @ S_N @ xx.T).item()
    print(f"var prédictive {lbl} : {pred_var:.4f}")
print("Plus on est loin des données, plus la prédiction est incertaine.")

### Exercice corrigé

In [ ]:
# ---- Exercice P7 : moments de Y = X^2, X~N(0,1) (changement de variable) ----
E = stats.norm(0,1).moment(2); V = stats.norm(0,1).moment(4) - E**2
print(f"P7-ex : E[Y] = {E:.1f}, Var[Y] = {V:.1f}")
assert np.isclose(E, 1.0) and np.isclose(V, 2.0)
print("\n✅ T05 termine — 7 points, exercices integres.")
